In [1]:
# 1. Install Redis Server system package and Python clients
!apt-get install redis-server -y > /dev/null
!pip install redis aiohttp beautifulsoup4 lxml

# 2. Launch the Redis engine silently in the background
!redis-server --daemonize yes

# 3. Test the connection to ensure it is live
import redis
r = redis.Redis(host='localhost', port=6379)
if r.ping():
    print("[+] Success! Your background Redis server is up and listening.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 12.7 MB/s eta 0:00:00
[+] Success! Your background Redis server is up and listening.


In [2]:
import asyncio
import aiohttp
import sqlite3
import redis
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# --- 1. SET UP GLOBAL CONFIGS ---
USER_AGENT = "ColabSDECrawler/1.0"
REQUEST_DELAY = 1.0  # Be polite: Wait 1 second between requests
TIMEOUT = 10         # Drop the connection if it hangs for 10 seconds
QUEUE_KEY = "crawler:frontier"
VISITED_KEY = "crawler:visited"

# Initialize Redis Client to talk to our background server
redis_client = redis.Redis(host="localhost", port=6379, decode_responses=True)

# --- 2. RELATIONAL DATA STORE LAYER ---
def init_db():
    """Creates a local SQLite database file to hold structured metrics."""
    conn = sqlite3.connect("scraped_data.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS web_pages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            url TEXT UNIQUE,
            title TEXT,
            description TEXT,
            scraped_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

def save_page_metadata(url, title, description):
    """Safely saves unique data records without duplicates."""
    try:
        conn = sqlite3.connect("scraped_data.db")
        cursor = conn.cursor()
        cursor.execute(
            "INSERT OR IGNORE INTO web_pages (url, title, description) VALUES (?, ?, ?)",
            (url, title, description)
        )
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"[-] Database Error: {e}")

# --- 3. THE TEXT & CONTENT PARSER ---
def parse_html(html_content, base_url):
    """Parses raw HTML text, extracts metadata, and extracts future link pathways."""
    soup = BeautifulSoup(html_content, "lxml")

    # Extract structural components
    title = soup.title.string.strip() if soup.title else "No Title"
    desc_tag = soup.find("meta", attrs={"name": "description"})
    description = desc_tag["content"].strip() if desc_tag and desc_tag.get("content") else "No Description Provided"

    discovered_links = set()
    # Scrape all hyper-anchors
    for anchor in soup.find_all("a", href=True):
        # Resolve absolute pathways (e.g., /docs -> https://example.com/docs)
        absolute_url = urljoin(base_url, anchor["href"])
        parsed_url = urlparse(absolute_url)
        clean_url = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}"

        if parsed_url.scheme in ["http", "https"]:
            discovered_links.add(clean_url)

    return title, description, list(discovered_links)

# --- 4. CORE ENGINE MULTI-THREADED MANAGER ---
class ColabCrawler:
    def __init__(self, seed_urls, max_workers=3):
        self.max_workers = max_workers
        init_db()
        redis_client.flushall() # Clean previous states out of Redis

        # Hydrate our frontier queue with root nodes
        for url in seed_urls:
            redis_client.sadd(QUEUE_KEY, url)

    async def fetch(self, session, url):
        """Executes a non-blocking HTTP GET network request."""
        headers = {"User-Agent": USER_AGENT}
        try:
            async with session.get(url, headers=headers, timeout=TIMEOUT) as response:
                if response.status == 200 and "text/html" in response.headers.get("Content-Type", ""):
                    return await response.text()
        except:
            return None # Gracefully skip request dropouts

    async def worker(self, worker_id):
        """Worker lifecycle loop. Pops URLs from Redis and coordinates the pipeline."""
        async with aiohttp.ClientSession() as session:
            while True:
                # Atomically grab a URL from the shared queue
                url = redis_client.spop(QUEUE_KEY)
                if not url:
                    await asyncio.sleep(2) # Backoff if queue is empty
                    continue

                # Deduplication check
                if redis_client.sismember(VISITED_KEY, url):
                    continue

                print(f"[Worker {worker_id}] Crawling -> {url}")
                redis_client.sadd(VISITED_KEY, url)
                await asyncio.sleep(REQUEST_DELAY) # Politeness pacing

                # Run Network Request
                html = await self.fetch(session, url)
                if not html:
                    continue

                # Run Parsing and extraction
                title, desc, links = parse_html(html, url)
                save_page_metadata(url, title, desc)

                # Feed newly discovered links back into the Redis queue
                for link in links:
                    if not redis_client.sismember(VISITED_KEY, link):
                        redis_client.sadd(QUEUE_KEY, link)

    async def run(self):
        # Fire off concurrent tracking paths across our worker count
        workers = [asyncio.create_task(self.worker(i)) for i in range(self.max_workers)]
        await asyncio.gather(*workers)

# --- 5. INITIALIZE TRACKING ---
# We use open tech portals as safe example seeds
seeds = ["https://news.ycombinator.com/", "https://example.com"]
crawler = ColabCrawler(seed_urls=seeds, max_workers=3)

print("[*] Starting the Crawler engine loop. Press the STOP icon anytime to halt.")
await crawler.run()

[*] Starting the Crawler engine loop. Press the STOP icon anytime to halt.
[Worker 0] Crawling -> https://example.com
[Worker 1] Crawling -> https://news.ycombinator.com/
[Worker 0] Crawling -> https://iana.org/domains/example
[Worker 1] Crawling -> https://brooker.co.za/blog/2020/08/06/erlang.html
[Worker 2] Crawling -> https://startupfortune.com/hyundai-takes-full-control-of-boston-dynamics-as-softbank-exits-for-325-million/
[Worker 0] Crawling -> https://iana.org/domains
[Worker 1] Crawling -> https://news.ycombinator.com/ask
[Worker 0] Crawling -> http://brooker.co.za/blog/2017/12/28/mean.html
[Worker 1] Crawling -> https://brooker.co.za/blog
[Worker 2] Crawling -> https://iana.org/domains/root/db
[Worker 1] Crawling -> https://metiq.space
[Worker 0] Crawling -> https://brooker.co.za/blog/2014/03/30/lamport-pub.html
[Worker 2] Crawling -> https://brooker.co.za/blog/2020/03/22/rust.html
[Worker 0] Crawling -> https://iana.org/domains/root/db/nz.html
[Worker 1] Crawling -> https://ia

CancelledError: 

In [3]:
import pandas as pd
import sqlite3

# Query the database file
conn = sqlite3.connect("scraped_data.db")
df = pd.read_sql_query("SELECT id, title, url FROM web_pages", conn)
conn.close()

print(f"🎉 Success! Total Unique Web Pages Saved: {len(df)}")
# View the first 10 rows
df.head(10)

🎉 Success! Total Unique Web Pages Saved: 30


,id,title,url
0,1,Example Domain,https://example.com
1,2,Hacker News,https://news.ycombinator.com/
2,3,Example Domains,https://iana.org/domains/example
3,4,Surprising Economics of Load-Balanced Systems ...,https://brooker.co.za/blog/2020/08/06/erlang.html
4,5,Domain Name Services,https://iana.org/domains
5,6,Ask | Hacker News,https://news.ycombinator.com/ask
6,7,Hyundai takes full control of Boston Dynamics ...,https://startupfortune.com/hyundai-takes-full-...
7,8,Marc Brooker's Blog - Marc's Blog,https://brooker.co.za/blog
8,9,Is the Mean Really Useless? - Marc's Blog,http://brooker.co.za/blog/2017/12/28/mean.html
9,10,Root Zone Database,https://iana.org/domains/root/db


In [4]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("scraped_data.db")

# 1. Total unique URL pathways logged
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM web_pages")
total_urls = cursor.fetchone()[0]

# 2. Calculate pages per minute
df = pd.read_sql_query("SELECT scraped_at FROM web_pages", conn)
conn.close()

if len(df) > 1:
    df['scraped_at'] = pd.to_datetime(df['scraped_at'])
    total_time_minutes = (df['scraped_at'].max() - df['scraped_at'].min()).total_seconds() / 60

    # Avoid dividing by zero if it ran incredibly fast
    if total_time_minutes == 0: total_time_minutes = 0.1

    pages_per_minute = int(len(df) / total_time_minutes)
else:
    pages_per_minute = 0

print("------ YOUR RESUME METRICS ------")
print(f"Total URL Pathways Logged: {total_urls}")
print(f"Estimated Pages Per Minute: {pages_per_minute}")

------ YOUR RESUME METRICS ------
Total URL Pathways Logged: 30
Estimated Pages Per Minute: 90
